# DL Colab Train-Only

This notebook retrains deep-learning checkpoints for NSL-KDD, UNSW-NB15, and CICIDS2017 on Google Colab. It prepares data, verifies file readability, preprocesses each dataset once, trains the configured DL models, saves per-run JSON outputs/checkpoints, and exports a Drive backup zip.

It intentionally does **not** run final cross-model evaluation/report synthesis. That comes after all new artifacts exist.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print(f"Drive mount skipped or unavailable: {exc}")

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/HoangZuzi-14/Nids_deep_learning.git"
CLONE_DIR = Path("/content/Nids_deep_learning")
DRIVE_PROJECT_CANDIDATES = [
    Path("/content/drive/MyDrive/nids_deep_learning/ids_deep_learning"),
    Path("/content/drive/MyDrive/Nids_deep_learning/ids_deep_learning"),
]


def is_project_dir(path: Path) -> bool:
    return (path / "src").exists() and (path / "config").exists()


def find_project_dir():
    candidates = [
        *DRIVE_PROJECT_CANDIDATES,
        Path.cwd(),
        Path.cwd() / "ids_deep_learning",
        CLONE_DIR,
        CLONE_DIR / "ids_deep_learning",
        Path("/content/ids_deep_learning"),
        Path("/content/ids_deep_learning") / "ids_deep_learning",
    ]
    for candidate in candidates:
        if is_project_dir(candidate):
            return candidate
    return None


PROJECT_DIR = find_project_dir()
if PROJECT_DIR is None:
    subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)
    PROJECT_DIR = find_project_dir()

if PROJECT_DIR is None:
    raise RuntimeError("Could not locate project directory containing src/ and config/.")

for candidate in [PROJECT_DIR, PROJECT_DIR.parent, CLONE_DIR]:
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

os.chdir(PROJECT_DIR)
print(f"Project directory: {PROJECT_DIR}")
print("Python search path head:", sys.path[:4])

In [ ]:
%pip install -q -r requirements.txt

## Configuration

Default settings run all three datasets and all four DL architectures. For a quick smoke run, set `DATASETS_TO_RUN = ["nsl_kdd"]`, reduce `EPOCHS`, or set `SAMPLE_FRACTION_BY_DATASET["cicids2017"] = 0.25` before running the training cell.

In [ ]:
from pathlib import Path

DATASETS_TO_RUN = ["nsl_kdd", "unsw_nb15", "cicids2017"]
CLASSIFICATION = "multi"

DL_EXPERIMENTS = [
    {"experiment_name": "MLP_Modular", "model_name": "mlp"},
    {"experiment_name": "CNN1D_Modular", "model_name": "cnn1d"},
    {"experiment_name": "BiLSTMAttention_Modular", "model_name": "bilstm"},
    {"experiment_name": "CNNLSTMHybrid_Modular", "model_name": "hybrid"},
]

EPOCHS = 30
BATCH_SIZE = 512
LEARNING_RATE = 1e-3
PATIENCE = 10
USE_FOCAL_LOSS = False
USE_WEIGHTED_SAMPLER = False
FOCAL_GAMMA = 2.0
STOP_ON_ERROR = True
SAVE_PROGRESS_TO_DRIVE = True

SAMPLE_FRACTION_BY_DATASET = {
    "nsl_kdd": 1.0,
    "unsw_nb15": 1.0,
    "cicids2017": 1.0,
}

DATA_ROOT = Path("/content/nids_data")
NSL_PATH = DATA_ROOT / "nsl_kdd" / "KDDTrain+.txt"
UNSW_PATH = DATA_ROOT / "unsw_nb15" / "training-set.csv"
CICIDS_CACHE_PATH = Path("/content/cicids2017_cache/merged.csv")
OUTPUT_DIR = Path("/content/drive/MyDrive/nids_outputs/dl_train_only")
OUTPUT_ZIP = Path("/content/drive/MyDrive/nids_outputs/dl_train_only_artifacts.zip")

print("Datasets:", DATASETS_TO_RUN)
print("Experiments:", DL_EXPERIMENTS)
print("Epochs:", EPOCHS, "Batch size:", BATCH_SIZE)
print("Output dir:", OUTPUT_DIR)

In [ ]:
from dataclasses import dataclass
import shutil
import urllib.request
import zipfile


@dataclass
class FileProbeReport:
    path: Path
    exists: bool
    ok: bool
    stat_size: int = 0
    readable_size: int = 0
    line_count: int = 0
    message: str = ""


def probe_file(path: Path, chunk_size: int = 1024 * 1024):
    path = Path(path)
    if not path.exists():
        return FileProbeReport(path=path, exists=False, ok=False, message="missing")
    try:
        stat_size = path.stat().st_size
        readable_size = 0
        line_count = 0
        with path.open("rb") as handle:
            for chunk in iter(lambda: handle.read(chunk_size), b""):
                readable_size += len(chunk)
                line_count += chunk.count(b"\n")
    except OSError as exc:
        return FileProbeReport(path=path, exists=True, ok=False, message=f"read error: {exc}")
    ok = readable_size == stat_size
    message = "ok" if ok else "readable bytes differ from filesystem stat size"
    return FileProbeReport(path, True, ok, stat_size, readable_size, line_count, message)


def require_readable_file(path: Path, label: str):
    report = probe_file(path)
    print(
        f"{label}: ok={report.ok} "
        f"stat_MB={report.stat_size / 1024 / 1024:.3f} "
        f"read_MB={report.readable_size / 1024 / 1024:.3f} "
        f"lines={report.line_count} | {path}"
    )
    if not report.ok:
        raise RuntimeError(f"File is missing or incomplete: {path} ({report.message})")
    return report


def download_if_missing(url: str, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        print(f"Downloading {url} -> {path}")
        urllib.request.urlretrieve(url, path)
    return require_readable_file(path, path.name)


def copy_preserve_relative(src: Path, root: Path, dest_root: Path):
    src = Path(src)
    if not src.exists():
        return None
    try:
        rel = src.relative_to(root)
    except ValueError:
        rel = Path(src.name)
    dest = dest_root / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dest)
    return dest

In [ ]:
import yaml

if "nsl_kdd" in DATASETS_TO_RUN:
    download_if_missing(
        "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt",
        NSL_PATH,
    )

if "unsw_nb15" in DATASETS_TO_RUN:
    download_if_missing(
        "https://raw.githubusercontent.com/ushukkla/nospammers/master/UNSW_NB15_training-set.csv",
        UNSW_PATH,
    )

config_path = PROJECT_DIR / "config" / "datasets.yaml"
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
if "nsl_kdd" in config.get("datasets", {}):
    config["datasets"]["nsl_kdd"]["cache_path"] = str(NSL_PATH)
if "unsw_nb15" in config.get("datasets", {}):
    config["datasets"]["unsw_nb15"]["cache_path"] = str(UNSW_PATH)
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

print("Configured NSL-KDD path:", config["datasets"]["nsl_kdd"]["cache_path"])
print("Configured UNSW-NB15 path:", config["datasets"]["unsw_nb15"]["cache_path"])

In [ ]:
EXPECTED_CICIDS_FILES = [
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
]

ALTERNATE_CICIDS_NAMES = {
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv": [
        "Friday-WorkingHours-Afternoon-DDoS.pcap_ISCX.csv",
        "Friday-WorkingHours-Afternoon-DDOS.pcap_ISCX.csv",
    ],
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv": [
        "Thursday-WorkingHours-Afternoon-Infiltration.pcap_ISCX.csv",
    ],
    "Wednesday-workingHours.pcap_ISCX.csv": [
        "Wednesday-WorkingHours.pcap_ISCX.csv",
        "Wednesday-workinghours.pcap_ISCX.csv",
    ],
}

CICIDS_CANDIDATE_RAW_DIRS = [
    Path("/content/drive/MyDrive/nids_deep_learning/ids_deep_learning/data/raw/cicids2017"),
    Path("/content/drive/MyDrive/nids_data/cicids2017/raw"),
    Path("/content/drive/MyDrive/cicids2017_raw"),
    Path("/content/cicids2017_raw"),
    PROJECT_DIR / "data" / "raw" / "cicids2017",
]
LOCAL_CICIDS_RAW_DIR = Path("/content/cicids2017_raw")
LOCAL_CICIDS_RAW_DIR.mkdir(parents=True, exist_ok=True)
CICIDS_RAW_DIR = None
PRESENT_CICIDS_FILES = {}


def cicids_candidate_names(expected_name):
    return [expected_name, *ALTERNATE_CICIDS_NAMES.get(expected_name, [])]


def find_cicids_present_files(raw_dir: Path):
    if not raw_dir.exists():
        return {}
    existing = {path.name.lower(): path for path in raw_dir.glob("*.csv")}
    present = {}
    for expected in EXPECTED_CICIDS_FILES:
        for name in cicids_candidate_names(expected):
            match = existing.get(name.lower())
            if match is not None:
                present[expected] = match
                break
    return present


def select_cicids_raw_dir():
    for candidate in CICIDS_CANDIDATE_RAW_DIRS:
        present = find_cicids_present_files(candidate)
        if len(present) == len(EXPECTED_CICIDS_FILES):
            return candidate, present
    return None, {}


def zip_member_map(zip_path: Path):
    with zipfile.ZipFile(zip_path) as archive:
        members = [info for info in archive.infolist() if not info.is_dir()]
    by_name = {Path(info.filename).name.lower(): info for info in members}
    result = {}
    for expected in EXPECTED_CICIDS_FILES:
        for name in cicids_candidate_names(expected):
            match = by_name.get(name.lower())
            if match is not None:
                result[expected] = match
                break
    return result


def extract_cicids_expected_files(zip_path: Path, target_dir: Path):
    target_dir.mkdir(parents=True, exist_ok=True)
    member_map = zip_member_map(zip_path)
    with zipfile.ZipFile(zip_path) as archive:
        for expected, info in member_map.items():
            with archive.open(info) as src, (target_dir / expected).open("wb") as dst:
                shutil.copyfileobj(src, dst)


if "cicids2017" in DATASETS_TO_RUN:
    CICIDS_RAW_DIR, PRESENT_CICIDS_FILES = select_cicids_raw_dir()
    print("Candidate CICIDS raw directories:")
    for candidate in CICIDS_CANDIDATE_RAW_DIRS:
        csv_count = len(list(candidate.glob("*.csv"))) if candidate.exists() else 0
        present_count = len(find_cicids_present_files(candidate)) if candidate.exists() else 0
        print(f"- {candidate} | exists={candidate.exists()} | csv_count={csv_count} | required_found={present_count}/8")

    if CICIDS_RAW_DIR is None:
        zip_candidates = []
        for base in [Path("/content"), Path("/content/drive/MyDrive"), Path("/content/drive/MyDrive/nids_data")]:
            if base.exists():
                zip_candidates.extend(sorted(base.glob("*.zip")))
        print("Zip candidates:", [str(path) for path in zip_candidates])
        for zip_path in zip_candidates:
            try:
                if len(zip_member_map(zip_path)) == len(EXPECTED_CICIDS_FILES):
                    print("Extracting CICIDS2017 raw CSV files from:", zip_path)
                    extract_cicids_expected_files(zip_path, LOCAL_CICIDS_RAW_DIR)
                    CICIDS_RAW_DIR, PRESENT_CICIDS_FILES = select_cicids_raw_dir()
                    break
            except zipfile.BadZipFile:
                pass

    if CICIDS_RAW_DIR is None:
        print("Missing CICIDS2017 files. Put the eight CSVs in one of these folders:")
        for candidate in CICIDS_CANDIDATE_RAW_DIRS:
            print("-", candidate)
        raise FileNotFoundError("CICIDS2017 raw CSV files are not available in Colab yet.")

    print("Selected CICIDS raw dir:", CICIDS_RAW_DIR)
    for expected, actual_path in PRESENT_CICIDS_FILES.items():
        require_readable_file(actual_path, expected)

    config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
    config["datasets"]["cicids2017"]["raw_dir"] = str(CICIDS_RAW_DIR)
    config["datasets"]["cicids2017"]["cache_path"] = str(CICIDS_CACHE_PATH)
    config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    print("Configured CICIDS2017:", config["datasets"]["cicids2017"])
else:
    print("CICIDS2017 not selected; skipping raw-file checks.")

In [ ]:
# Make the notebook resilient when Colab is using an older clone of the repo.
label_mapping_path = PROJECT_DIR / "src" / "models" / "preprocessing" / "label_mapping.py"
if label_mapping_path.exists():
    text = label_mapping_path.read_text(encoding="utf-8")
    old = "mapped = normalized.map(mapping or {}).fillna(normalized.str.title())"
    new = """normalized_mapping = {
        _normalize_label(source): target for source, target in (mapping or {}).items()
    }
    mapped = normalized.map(normalized_mapping).fillna(normalized.str.title())"""
    if old in text and "normalized_mapping =" not in text:
        label_mapping_path.write_text(text.replace(old, new), encoding="utf-8")
        print("Patched label_mapping.py to normalize mapping keys.")

import importlib
import pandas as pd
from src.preprocessing.label_mapping import NSL_KDD_MULTICLASS, map_multiclass_labels

encoded, _, mapping = map_multiclass_labels(pd.Series(["buffer_overflow", "guess_passwd"]), NSL_KDD_MULTICLASS)
inverse = {idx: label for label, idx in mapping.items()}
print("NSL mapping smoke:", [inverse[int(value)] for value in encoded])

In [ ]:
import gc
import json
import time
import traceback

import joblib
import numpy as np
import torch
import yaml

from src.pipeline.baseline_runner import build_adapter
from src.pipeline.dl_runner import run_dl_experiment_from_output
from src.utils.seed import set_seed

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_RESULTS_DIR = PROJECT_DIR / "results"
PROJECT_ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
PROJECT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = PROJECT_RESULTS_DIR / "dl_train_only_manifest.json"
drive_manifest_path = OUTPUT_DIR / "dl_train_only_manifest.json"
manifest = {
    "mode": "dl_train_only",
    "classification": CLASSIFICATION,
    "datasets_to_run": DATASETS_TO_RUN,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "patience": PATIENCE,
    "use_focal_loss": USE_FOCAL_LOSS,
    "use_weighted_sampler": USE_WEIGHTED_SAMPLER,
    "sample_fraction_by_dataset": SAMPLE_FRACTION_BY_DATASET,
    "device": str(DEVICE),
    "runs": [],
}


def save_manifest():
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    if SAVE_PROGRESS_TO_DRIVE:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        drive_manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")


def sync_run_outputs(dataset_key: str, experiment_name: str):
    if not SAVE_PROGRESS_TO_DRIVE:
        return []
    copied = []
    artifact_dir = PROJECT_DIR / "artifacts" / dataset_key / CLASSIFICATION
    result_path = PROJECT_DIR / "results" / f"{dataset_key}_{CLASSIFICATION}_{experiment_name}_results.json"
    shared_names = ["scaler.pkl", "encoders.pkl", "imputer.pkl", "label_mapping.json", "inference_config.json"]
    run_names = [f"{experiment_name}.pt", f"{experiment_name}_metadata.pkl"]
    for name in [*shared_names, *run_names]:
        copied_path = copy_preserve_relative(artifact_dir / name, PROJECT_DIR, OUTPUT_DIR)
        if copied_path is not None:
            copied.append(str(copied_path))
    copied_path = copy_preserve_relative(result_path, PROJECT_DIR, OUTPUT_DIR)
    if copied_path is not None:
        copied.append(str(copied_path))
    return copied


def preprocess_dataset_once(dataset_key: str):
    datasets_config = yaml.safe_load((PROJECT_DIR / "config" / "datasets.yaml").read_text(encoding="utf-8"))
    experiments_config = yaml.safe_load((PROJECT_DIR / "config" / "experiments.yaml").read_text(encoding="utf-8"))
    seed = int(experiments_config.get("seed", 42))
    set_seed(seed)
    split_cfg = experiments_config.get("split", {})
    adapter = build_adapter(dataset_key, datasets_config, PROJECT_DIR, seed)
    output = adapter.preprocess(
        classification_type=CLASSIFICATION,
        test_size=float(split_cfg.get("test_size", 0.2)),
        val_size=float(split_cfg.get("val_size", 0.2)),
        scaler_type=experiments_config.get("preprocessing", {}).get("scaler", "standard"),
    )
    return output


save_manifest()
for dataset_key in DATASETS_TO_RUN:
    print("=" * 80)
    print("Preprocessing dataset:", dataset_key)
    output = preprocess_dataset_once(dataset_key)
    dataset_info = {
        "dataset": dataset_key,
        "feature_count": int(output.X_train.shape[1]),
        "label_mapping": output.label_mapping,
        "split_shapes": {
            "train": list(output.X_train.shape),
            "val": list(output.X_val.shape),
            "test": list(output.X_test.shape),
        },
    }
    print(json.dumps(dataset_info, indent=2))

    for experiment in DL_EXPERIMENTS:
        experiment_name = experiment["experiment_name"]
        model_name = experiment["model_name"]
        run_record = {
            "dataset": dataset_key,
            "experiment": experiment_name,
            "model_name": model_name,
            "status": "started",
            "started_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        }
        manifest["runs"].append(run_record)
        save_manifest()
        print("-" * 80)
        print(f"Training {dataset_key} / {experiment_name} ({model_name})")
        try:
            report = run_dl_experiment_from_output(
                output=output,
                dataset_key=dataset_key,
                classification_type=CLASSIFICATION,
                root=PROJECT_DIR,
                experiment_name=experiment_name,
                model_name=model_name,
                batch_size=BATCH_SIZE,
                epochs=EPOCHS,
                lr=LEARNING_RATE,
                patience=PATIENCE,
                use_focal_loss=USE_FOCAL_LOSS,
                use_weighted_sampler=USE_WEIGHTED_SAMPLER,
                focal_gamma=FOCAL_GAMMA,
                sample_fraction=float(SAMPLE_FRACTION_BY_DATASET.get(dataset_key, 1.0)),
            )
            metrics = report["models"][experiment_name]
            copied = sync_run_outputs(dataset_key, experiment_name)
            run_record.update(
                {
                    "status": "completed",
                    "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "accuracy": metrics.get("accuracy"),
                    "macro_f1": metrics.get("macro_f1"),
                    "weighted_f1": metrics.get("weighted_f1"),
                    "far": metrics.get("far"),
                    "checkpoint": metrics.get("checkpoint"),
                    "train_seconds": metrics.get("train_seconds"),
                    "copied_outputs": copied,
                }
            )
            print(
                f"Done {dataset_key}/{experiment_name}: "
                f"macro_f1={metrics.get('macro_f1'):.6f}, "
                f"far={metrics.get('far'):.6f}, "
                f"seconds={metrics.get('train_seconds'):.1f}"
            )
        except Exception as exc:
            run_record.update(
                {
                    "status": "failed",
                    "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "error": repr(exc),
                    "traceback": traceback.format_exc(),
                }
            )
            print("FAILED:", dataset_key, experiment_name, repr(exc))
            print(run_record["traceback"])
            if STOP_ON_ERROR:
                save_manifest()
                raise
        finally:
            save_manifest()
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

print("Training loop finished. Manifest:", manifest_path)
if SAVE_PROGRESS_TO_DRIVE:
    print("Drive manifest:", drive_manifest_path)

In [ ]:
import json

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
completed = [run for run in manifest["runs"] if run.get("status") == "completed"]
failed = [run for run in manifest["runs"] if run.get("status") == "failed"]
print("Completed runs:", len(completed))
print("Failed runs:", len(failed))

for run in completed:
    artifact_dir = PROJECT_DIR / "artifacts" / run["dataset"] / CLASSIFICATION
    ckpt_path = artifact_dir / f"{run['experiment']}.pt"
    meta_path = artifact_dir / f"{run['experiment']}_metadata.pkl"
    result_path = PROJECT_DIR / "results" / f"{run['dataset']}_{CLASSIFICATION}_{run['experiment']}_results.json"
    for label, path in [("checkpoint", ckpt_path), ("metadata", meta_path), ("result", result_path)]:
        report = probe_file(path)
        print(run["dataset"], run["experiment"], label, report.ok, report.readable_size)
        if not report.ok:
            raise RuntimeError(f"Incomplete {label}: {path}")

if failed:
    print("Failed run details are stored in:", manifest_path)
else:
    print("All configured DL runs completed and verified.")

In [ ]:
import zipfile

OUTPUT_ZIP.parent.mkdir(parents=True, exist_ok=True)
if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()

with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    if OUTPUT_DIR.exists():
        for path in sorted(OUTPUT_DIR.rglob("*")):
            if path.is_file() and path != OUTPUT_ZIP:
                archive.write(path, path.relative_to(OUTPUT_DIR.parent))
    if manifest_path.exists():
        archive.write(manifest_path, manifest_path.relative_to(PROJECT_DIR))

print("Wrote:", OUTPUT_ZIP)
print("Size MB:", OUTPUT_ZIP.stat().st_size / 1024 / 1024)

## Next Step After This Notebook

After all DL runs complete and `dl_train_only_artifacts.zip` exists in Drive, run the final evaluation notebook/script. That later step should compare the newly retrained ML and DL outputs together; this notebook only prepares the artifacts and per-run result JSON files.